In [1]:
import os
import sys
import datetime as dt
import cmocean as cmo

import cartopy.crs as ccrs
import easygems.healpix as egh
import intake
import matplotlib.pyplot as plt
import numpy as np

import healpy as hp
import xarray as xr

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# Filter out annoying warning.
import warnings

from utils import haversine, hp_mods, get_nn_lon_lat_index, mass_weighted_column_integral

warnings.filterwarnings(
    "ignore",
    message=".*The return type of `Dataset.dims` will be changed.*",
    category=FutureWarning,
)

In [3]:
# Open catalog.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
sim = 'um_glm_n2560_RAL3p3_tuned_hk26'
# sim = 'um_glm_n1280_GAL9_v2_hk26'
sim_cat = cat[sim]
ds = sim_cat(zoom=3, time='PT3H').to_dask().pipe(hp_mods)
ds

<xarray.Dataset> Size: 3GB
Dimensions:   (time: 3249, pressure: 25, cell: 768)
Coordinates:
  * pressure  (pressure) int64 200B 1 5 10 20 30 50 ... 875 900 925 950 975 1000
  * time      (time) datetime64[ns] 26kB 2020-01-20 ... 2021-03-01
    crs       int64 8B 0
  * cell      (cell) int64 6kB 0 1 2 3 4 5 6 7 ... 761 762 763 764 765 766 767
    lat       (cell) float64 6kB 4.78 9.594 9.594 14.48 ... -9.594 -9.594 -4.78
    lon       (cell) float64 6kB 45.0 50.62 39.38 45.0 ... 320.6 309.4 315.0
Data variables: (12/14)
    cli       (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    clw       (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    hur       (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    hus       (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    orog      (cell) float64 6kB dask.array<chunksize=(768,), meta=np.ndarray>
    qg        (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    ...        ...
    sftlf     (cell) float64 6kB dask.array<chunksize=(768,), meta=np.ndarray>
    ta        (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    ua        (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    va        (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    wa        (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
    zg        (time, pressure, cell) float32 250MB dask.array<chunksize=(1024, 5, 768), meta=np.ndarray>
Attributes:
    regional_bounds:         {'lower_left_lat': -90, 'lower_left_lon': 0, 'up...
    latitiude_convention:    [-90, 90]
    longitude_convention:    [0, 360]
    regional:                False
    simulation:              glm.n2560_RAL3p3.tuned
    simulation_description:  The MetUM uses a regular lat-lon grid, for our e...
    processing_version:      v7
    deploy:                  prod
    summary:                 Met Office DYAMOND3 simulations: A group of expe...
    Conventions:             CF-1.13

In [ ]:
import metpy.constants as mpconst

g = mpconst.g.magnitude
cp = mpconst.dry_air_spec_heat_press.magnitude
Lv = mpconst.water_heat_vaporization.magnitude

# compute h (J/kg)
h = cp * ds.ta + ds.zg + Lv * ds.hus
H = mass_weighted_column_integral(h)

egh.healpix_show(H.sel(time="2020-07-10 12:00"))